# Per-Finger Touch Detection Model (LSTM in PyTorch)

This notebook loads the CSV datasets (`./data/training_data.csv` and `./data/test_data.csv`), prepares sequence feature matrices from the **4 transition steps x 8 velocity dimensions** per row, and trains/evaluates the **PyTorch LSTM touch detector** following the initial blueprint in `touch_lstm_model.py`.

## 1. Imports & Hyperparameters Setup (Original Baseline Setup)

In [34]:
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

# Set random seeds for reproducibility
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# Original Baseline Hyperparameters
SEQ_LEN = 4          # 4 velocity transition steps (1..4)
FEATURE_DIM = 8      # 2 wrist vels + 6 finger joint vels (mcp_vx, mcp_vy, pip_vx, pip_vy, dip_vx, dip_vy)
BATCH_SIZE = 32
LEARNING_RATE = 0.001
HIDDEN_UNITS = 32     # Original 32 hidden units
DROPOUT = 0.2         # Original 0.2 dropout
EPOCHS = 100

# Setup device agnostic code
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## 2. Load & Prepare CSV Datasets (`training_data.csv` & `test_data.csv`)

Each row in `training_data.csv` and `test_data.csv` represents a single per-finger 5-frame window and contains **4 transition steps** (step 1..4).
For each step $v \in \{1, 2, 3, 4\}$, the 8 velocity features extracted are:
- Wrist: `wrist{v}_vx`, `wrist{v}_vy`
- MCP: `mcp{v}_vx`, `mcp{v}_vy`
- PIP: `pip{v}_vx`, `pip{v}_vy`
- DIP: `dip{v}_vx`, `dip{v}_vy`

In [35]:
def extract_features_and_labels(csv_path):
    df = pd.read_csv(csv_path)
    
    # Define feature column names for sequence of 4 timesteps x 8 features
    # Each timestep v (1..4) has 8 velocity components
    vel_cols_step = []
    for v in range(1, 5):
        cols = [
            f"wrist{v}_vx", f"wrist{v}_vy",
            f"mcp{v}_vx", f"mcp{v}_vy",
            f"pip{v}_vx", f"pip{v}_vy",
            f"dip{v}_vx", f"dip{v}_vy"
        ]
        vel_cols_step.append(cols)
        
    n_samples = len(df)
    X = np.zeros((n_samples, 4, 8), dtype=np.float32)
    
    for step_idx in range(4):
        cols = vel_cols_step[step_idx]
        X[:, step_idx, :] = df[cols].fillna(0.0).values.astype(np.float32)
        
    # Extract target label (touch_finger or touch)
    target_col = "touch_finger" if "touch_finger" in df.columns else "touch"
    y = df[target_col].astype(str).str.strip().str.lower().isin(["1", "true", "t", "yes", "y"]).values.astype(np.float32)
    y = y.reshape(-1, 1)
    
    return X, y

TRAIN_CSV = "./data/training_data.csv"
TEST_CSV = "./data/test_data.csv"

X_train_np, y_train_np = extract_features_and_labels(TRAIN_CSV)
X_test_np, y_test_np = extract_features_and_labels(TEST_CSV)

# Convert to PyTorch Tensors
X_train = torch.from_numpy(X_train_np).type(torch.float32)
y_train = torch.from_numpy(y_train_np).type(torch.float32)
X_test = torch.from_numpy(X_test_np).type(torch.float32)
y_test = torch.from_numpy(y_test_np).type(torch.float32)

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test.shape},  y_test shape:  {y_test.shape}")

X_train shape: torch.Size([2765, 4, 8]), y_train shape: torch.Size([2765, 1])
X_test shape:  torch.Size([6527, 4, 8]),  y_test shape:  torch.Size([6527, 1])


## 3. PyTorch Dataset and DataLoader

In [36]:
class VelocitySequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = VelocitySequenceDataset(X_train, y_train)
test_dataset = VelocitySequenceDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

## 4. Building the Original LSTM Touch Detection Model

In [37]:
class FingerTouchLSTM(nn.Module):
    def __init__(self, input_features: int = 8, hidden_units: int = 32, num_layers: int = 2, dropout: float = 0.2):
        super().__init__()
        
        # LSTM Layer processing (batch_size, seq_len, input_features)
        self.lstm = nn.LSTM(
            input_size=input_features,
            hidden_size=hidden_units,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        
        # Fully connected classifier head for binary touch prediction
        self.classifier = nn.Sequential(
            nn.Linear(in_features=hidden_units, out_features=16),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(in_features=16, out_features=1)
        )
        
    def forward(self, x):
        # Forward pass through LSTM: lstm_out shape -> (batch_size, seq_len, hidden_units)
        lstm_out, (hn, cn) = self.lstm(x)
        
        # Select output from the final sequence timestep (t = seq_len - 1)
        last_timestep = lstm_out[:, -1, :]
        
        # Pass unnormalized logits to classifier head
        logits = self.classifier(last_timestep)
        return logits

# Instantiate original baseline model
model_0 = FingerTouchLSTM(input_features=FEATURE_DIM, hidden_units=HIDDEN_UNITS, num_layers=2, dropout=DROPOUT).to(device)
print(model_0)

FingerTouchLSTM(
  (lstm): LSTM(8, 32, num_layers=2, batch_first=True, dropout=0.2)
  (classifier): Sequential(
    (0): Linear(in_features=32, out_features=16, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=16, out_features=1, bias=True)
  )
)


## 5. Loss Function, Optimizer, and Accuracy Function

In [38]:
# Loss Function & Optimizer (Original Adam setup)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(params=model_0.parameters(), lr=LEARNING_RATE)

# Accuracy Calculation Function
def accuracy_fn(y_true, y_pred):
    correct = torch.eq(y_true, y_pred).sum().item()
    acc = (correct / len(y_pred)) * 100
    return acc

## 6. Initial Un-trained Model Evaluation

In [39]:
model_0.eval()
with torch.inference_mode():
    sample_X = X_train[:5].to(device)
    sample_y = y_train[:5].to(device)
    
    y_logits = model_0(sample_X)
    y_pred_probs = torch.sigmoid(y_logits)
    y_pred_labels = torch.round(y_pred_probs)

print("Sample Raw Logits:\n", y_logits.squeeze())
print("Sample Prediction Probabilities:\n", y_pred_probs.squeeze())
print("Sample Predicted Labels:\n", y_pred_labels.squeeze())
print("True Target Labels:\n", sample_y.squeeze())

Sample Raw Logits:
 tensor([-0.0062, -0.0084, -0.0045, -0.0095, -0.0133], device='cuda:0')
Sample Prediction Probabilities:
 tensor([0.4985, 0.4979, 0.4989, 0.4976, 0.4967], device='cuda:0')
Sample Predicted Labels:
 tensor([0., 0., 0., 0., 0.], device='cuda:0')
True Target Labels:
 tensor([1., 0., 1., 1., 0.], device='cuda:0')


## 7. Training and Testing Loop (Original Version)

In [40]:
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed(RANDOM_SEED)

train_losses, test_losses = [], []
train_accuracies, test_accuracies = [], []

for epoch in range(1, EPOCHS + 1):
    # --- Training Phase ---
    model_0.train()
    train_loss, train_acc = 0.0, 0.0
    
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        y_logits = model_0(X_batch)
        loss = loss_fn(y_logits, y_batch)
        y_preds = torch.round(torch.sigmoid(y_logits))
        acc = accuracy_fn(y_batch, y_preds)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * len(X_batch)
        train_acc += (acc / 100.0) * len(X_batch)
        
    train_loss /= len(train_dataset)
    train_acc = (train_acc / len(train_dataset)) * 100
    train_losses.append(train_loss)
    train_accuracies.append(train_acc)
    
    # --- Testing Phase ---
    model_0.eval()
    test_loss, test_acc = 0.0, 0.0
    with torch.inference_mode():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            test_logits = model_0(X_batch)
            t_loss = loss_fn(test_logits, y_batch)
            test_preds = torch.round(torch.sigmoid(test_logits))
            t_acc = accuracy_fn(y_batch, test_preds)
            
            test_loss += t_loss.item() * len(X_batch)
            test_acc += (t_acc / 100.0) * len(X_batch)
            
        test_loss /= len(test_dataset)
        test_acc = (test_acc / len(test_dataset)) * 100
        test_losses.append(test_loss)
        test_accuracies.append(test_acc)
        
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch: {epoch:02d} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2f}%")

Epoch: 01 | Train Loss: 0.6510 | Train Acc: 63.29% | Test Loss: 0.4978 | Test Acc: 86.70%
Epoch: 05 | Train Loss: 0.3822 | Train Acc: 84.74% | Test Loss: 0.2938 | Test Acc: 84.25%
Epoch: 10 | Train Loss: 0.3475 | Train Acc: 85.50% | Test Loss: 0.3329 | Test Acc: 82.83%
Epoch: 15 | Train Loss: 0.3239 | Train Acc: 87.38% | Test Loss: 0.2763 | Test Acc: 85.05%
Epoch: 20 | Train Loss: 0.3096 | Train Acc: 87.81% | Test Loss: 0.2985 | Test Acc: 85.17%
Epoch: 25 | Train Loss: 0.2985 | Train Acc: 88.46% | Test Loss: 0.2535 | Test Acc: 87.59%
Epoch: 30 | Train Loss: 0.2958 | Train Acc: 89.08% | Test Loss: 0.3291 | Test Acc: 84.10%
Epoch: 35 | Train Loss: 0.2857 | Train Acc: 88.90% | Test Loss: 0.2959 | Test Acc: 85.25%
Epoch: 40 | Train Loss: 0.2810 | Train Acc: 89.04% | Test Loss: 0.2638 | Test Acc: 86.98%
Epoch: 45 | Train Loss: 0.2697 | Train Acc: 89.66% | Test Loss: 0.2805 | Test Acc: 86.47%
Epoch: 50 | Train Loss: 0.2725 | Train Acc: 89.73% | Test Loss: 0.2901 | Test Acc: 85.89%
Epoch: 55 

## 8. Plotting Training and Testing Curves

In [8]:
plt.figure(figsize=(12, 5))

# Plot Loss
plt.subplot(1, 2, 1)
plt.plot(range(1, EPOCHS + 1), train_losses, label="Train Loss", color="blue")
plt.plot(range(1, EPOCHS + 1), test_losses, label="Test Loss", color="red", linestyle="--")
plt.title("Loss Curves")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()

# Plot Accuracy
plt.subplot(1, 2, 2)
plt.plot(range(1, EPOCHS + 1), train_accuracies, label="Train Accuracy", color="blue")
plt.plot(range(1, EPOCHS + 1), test_accuracies, label="Test Accuracy", color="red", linestyle="--")
plt.title("Accuracy Curves")
plt.xlabel("Epochs")
plt.ylabel("Accuracy (%)")
plt.legend()

plt.tight_layout()
plt.show()

## 9. Single Sequence Inference Helper Function

In [9]:
def predict_touch(model, velocity_window_4x8):
    """
    Predicts touch event for a single 4x8 velocity window sequence.
    
    Args:
        velocity_window_4x8 (list or np.ndarray): Shape (4, 8) containing:
            [wrist_vx, wrist_vy, mcp_vx, mcp_vy, pip_vx, pip_vy, dip_vx, dip_vy] across 4 transition steps.
            
    Returns:
        tuple: (prob_score (float), is_touch (bool))
    """
    model.eval()
    tensor_in = torch.tensor(velocity_window_4x8, dtype=torch.float32).unsqueeze(0).to(device)
    
    with torch.inference_mode():
        logits = model(tensor_in)
        prob = torch.sigmoid(logits).item()
        
    is_touch = prob >= 0.5
    return prob, is_touch

# Test with a single sample from test set
sample_seq = X_test[0].numpy()
prob, is_touch = predict_touch(model_0, sample_seq)
print(f"Predicted Touch Probability: {prob:.4f} -> Is Touch: {is_touch}")

## 10. Saving Model Weights

In [10]:
MODEL_SAVE_PATH = "finger_touch_lstm.pth"
torch.save(model_0.state_dict(), MODEL_SAVE_PATH)
print(f"Model saved successfully to: {MODEL_SAVE_PATH}")